In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
import glob
from torch.utils.data import DataLoader

# Custom Dataset Class
class SUIMSegmentationDataset(Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):
        #self.root_dir = root_dir
        self.transform = transform
        self.target_transform = target_transform
        root_dir = os.path.join(path, "dataset")
        self.all_images = sorted(glob.glob(f"{root_dir}/images/*.jpg"))
        self.all_masks  = sorted(glob.glob(f"{root_dir}/masks/*.png"))

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_path = self.all_images[idx]
        mask_path = self.all_masks[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)


        mask = remap_mask(mask)

        return image, mask

image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),

])


In [ ]:

dataset = SUIMSegmentationDataset(path, image_transforms, mask_transforms)
dataset_loader = DataLoader(dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True)

import matplotlib.pyplot as plt

# Function to denormalize images (We cannot show normalized images. We have to reverse normalizaion first.)
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Display some images with their masks
for i in range(3):
    img, mask = dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1,2,0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()


In [ ]:
# TO DO

In [ ]:
# TO DO

In [ ]:
# TO DO

In [ ]:
# TO DO